In [ ]:
import os

os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

import tempfile
import uuid
from pathlib import Path

import torch
import yaml
from transformers import AutoTokenizer

from vllm_omni import AsyncOmni
from vllm_omni.model_executor.models.qwen3_tts.configuration_qwen3_tts import (
    Qwen3TTSConfig,
)
from vllm_omni.model_executor.models.qwen3_tts_nv.qwen3_tts_talker_nv import (
    Qwen3TTSTalkerForConditionalGenerationNv,
)

assert Path(MODEL_DIR).is_dir(), f"Model dir not found: {MODEL_DIR}"
print(f"Model dir: {MODEL_DIR}")

## 1. Build a talker-only stage config

In [ ]:
MAX_MODEL_LEN = 2048
MAX_NUM_BATCHED_TOKENS = 512
MAX_DECODE_STEPS = 128  # short run — we only care that it doesn't crash.

stage_cfg = {
    "stage_args": [
        {
            "stage_id": 0,
            "stage_type": "llm",
            "is_comprehension": True,
            "final_output": True,
            "final_output_type": "latent",
            "runtime": {"devices": "0"},
            "engine_args": {
                "model_stage": "qwen3_tts",
                "max_num_seqs": 1,
                "model_arch": "Qwen3TTSTalkerForConditionalGenerationNv",
                "worker_type": "ar",
                "scheduler_cls": "vllm_omni.core.sched.omni_ar_scheduler.OmniARScheduler",
                "enforce_eager": False,
                "trust_remote_code": True,
                "async_scheduling": True,
                "enable_prefix_caching": False,
                "engine_output_type": "latent",
                "gpu_memory_utilization": 0.5,
                "distributed_executor_backend": "mp",
                "max_num_batched_tokens": MAX_NUM_BATCHED_TOKENS,
                "max_model_len": MAX_MODEL_LEN,
            },
            "default_sampling_params": {
                "temperature": 0.9,
                "top_k": 50,
                "max_tokens": MAX_DECODE_STEPS,
                "seed": 42,
                "detokenize": False,
                "repetition_penalty": 1.05,
                "stop_token_ids": [2150],
            },
        }
    ],
}

_tmp = tempfile.NamedTemporaryFile(
    mode="w", suffix=".yaml", prefix="talker_nv_demo_", delete=False,
)
yaml.dump(stage_cfg, _tmp, sort_keys=False)
_tmp.close()
STAGE_CFG_PATH = _tmp.name
print(f"Stage config: {STAGE_CFG_PATH}")

## 2. Launch the AsyncOmni engine

In [ ]:
omni = AsyncOmni(
    model=MODEL_DIR,
    stage_configs_path=STAGE_CFG_PATH,
    log_stats=False,
    stage_init_timeout=300,
)
print("Engine ready (talker-NV only, dummy weights)")

## 3. Build a request from ``{text, speaker, language}``

In [ ]:
TEXT = (
    "The quick brown fox jumps over the lazy dog near the riverbank."
)
SPEAKER = "vivian"
LANGUAGE = "English"
TASK_TYPE = "CustomVoice"

additional_information = {
    "task_type": [TASK_TYPE],
    "text": [TEXT],
    "language": [LANGUAGE],
    "speaker": [SPEAKER],
}

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR, trust_remote_code=True, padding_side="left",
)
hf_cfg = Qwen3TTSConfig.from_pretrained(MODEL_DIR, trust_remote_code=True)
talker_cfg = getattr(hf_cfg, "talker_config", None)

prompt_len = Qwen3TTSTalkerForConditionalGenerationNv.estimate_prompt_len_from_additional_information(
    additional_information=additional_information,
    task_type=TASK_TYPE,
    tokenize_prompt=lambda t: tokenizer(t, padding=False)["input_ids"],
    codec_language_id=getattr(talker_cfg, "codec_language_id", None),
    spk_is_dialect=getattr(talker_cfg, "spk_is_dialect", None),
)

prompt = {
    "prompt_token_ids": [0] * prompt_len,
    "additional_information": additional_information,
}
print(f"Estimated prompt length: {prompt_len}")
print(f"Speaker: {SPEAKER!r}  Language: {LANGUAGE!r}")
print(f"Text: {TEXT!r}")

## 4. Submit the request and count emitted frames

With dummy weights the emitted codec tokens are garbage — all we check is
that the engine drives ``preprocess`` → ``forward`` → ``postprocess`` for a
prefill and ``MAX_DECODE_STEPS`` decode steps without blowing up.

In [ ]:
async def run_request(prompt: dict):
    request_id = f"demo-nv-{uuid.uuid4().hex[:8]}"
    final_ro = None
    num_steps = 0
    async for stage_output in omni.generate(prompt, request_id=request_id):
        final_ro = stage_output
        num_steps += 1
    return final_ro, num_steps


final_ro, num_steps = await run_request(prompt)
assert final_ro is not None, "no output from engine"

mm = final_ro.multimodal_output or {}
audio_codes = mm.get("audio_codes")
token_ids = final_ro.outputs[0].token_ids if final_ro.outputs else []

print(f"Engine steps yielded       : {num_steps}")
print(f"Layer-0 tokens (token_ids) : {len(token_ids)}")
if isinstance(audio_codes, torch.Tensor):
    print(f"audio_codes shape          : {tuple(audio_codes.shape)}")
    print(f"codes dtype                : {audio_codes.dtype}")
else:
    print(f"audio_codes                : {audio_codes!r}")
print("OK — talker-NV ran end-to-end on dummy weights.")

In [ ]:
import matplotlib.pylab as plt

gen_codes = audio_codes[prompt_len:]
plt.imshow(gen_codes.T, aspect="auto")
plt.colorbar()
plt.show()

## 5. Cleanup

In [ ]:
omni.shutdown()
try:
    os.unlink(STAGE_CFG_PATH)
except OSError:
    pass
print("Engine shut down.")